In [13]:
# Inspecting the embryos
import zipfile, json, os, time, numpy as np, pandas as pd
from pathlib import Path
base=Path('.')
# Error: When I run this notebook on Google Colab and say this:
# base=Path('/mnt/data')
# there's an error because there's no directory with a name starting with '/mnt/data/'
files={'zebrafish':'f2f28d77-0c5c-40aa-aa62-509e40294760.zip','mouse':'5ffeadd4-da71-4a27-a895-13caea41400a.zip'}
report={}
for species,name in files.items():
 t=time.perf_counter();print('Processing',species,flush=True)
 with zipfile.ZipFile(base/name) as z:
  expr_file=next(n for n in z.namelist() if n.endswith('.tsv') and '/metadata_' not in n)
  df=pd.read_csv(z.open(expr_file),sep='	',index_col=0).astype(np.float32)
  # I can also run this notebook and say this:
  # df=pd.read_csv(z.open(expr_file),sep='\t',index_col=0).astype(np.float32)
  # Even so, the output will most likely be the same, and the code still works.
  arr=df.to_numpy(dtype=np.float32,copy=False)
  nan=np.isnan(arr).sum()
  finite=np.isfinite(arr).sum()
  negative=(arr<0).sum()
  zero=(arr==0).sum()
  rowmeans=np.nanmean(arr,axis=1)
  rowvar=np.nanvar(np.log1p(np.clip(arr,0,None)),axis=1)
  colmeans=np.nanmean(arr,axis=0)
  colsums=np.nansum(arr,axis=0)
  topids=df.index[np.argsort(rowvar)[-10:][::-1]].tolist()
  report[species]={
   'expression_file':expr_file,
   'samples':int(df.shape[1]),'genes':int(df.shape[0]),
   'unique_gene_ids':int(df.index.nunique()),'non_finite_values':int(arr.size-finite),
   'nan_values':int(nan),'negative_values':int(negative),
   'zero_values':int(zero),'zero_fraction':round(float(zero/arr.size),5),
   'entry_min':round(float(np.nanmin(arr)),6),'entry_max':round(float(np.nanmax(arr)),6),
   'gene_mean_quantiles':dict(zip(['p0','p25','p50','p75','p100'],np.quantile(rowmeans,[0,.25,.5,.75,1]).round(4).tolist())),
   'sample_mean_quantiles':dict(zip(['p0','p25','p50','p75','p100'],np.quantile(colmeans,[0,.25,.5,.75,1]).round(4).tolist())),
   'sample_sum_quantiles':dict(zip(['p0','p25','p50','p75','p100'],np.quantile(colsums,[0,.25,.5,.75,1]).round(2).tolist())),
   'top_10_log1p_variable_gene_ids':topids
  }
  # useful portable top gene list
  variances=pd.DataFrame({'ensembl_gene_id':df.index.to_numpy(),'mean_processed_expr':rowmeans,'log1p_variance':rowvar})
  variances.sort_values('log1p_variance',ascending=False).to_csv(base/f'{species}_gene_variability.csv',index=False)
  pd.DataFrame({'sample_id':df.columns,'mean_processed_expr':colmeans,'sum_processed_expr':colsums}).to_csv(base/f'{species}_sample_summary.csv',index=False)
  del df,arr
 print('Completed',species,round(time.perf_counter()-t,1),'s',report[species],flush=True)
with open(base/'embryo_dataset_qc.json','w') as f:json.dump(report,f,indent=2)

Processing zebrafish
Completed zebrafish 13.0 s {'expression_file': 'SRP055996/SRP055996.tsv', 'samples': 1138, 'genes': 31882, 'unique_gene_ids': 31882, 'non_finite_values': 0, 'nan_values': 0, 'negative_values': 0, 'zero_values': 0, 'zero_fraction': 0.0, 'entry_min': 0.008134, 'entry_max': 4.976743, 'gene_mean_quantiles': {'p0': 0.1813, 'p25': 0.1915, 'p50': 0.2428, 'p75': 0.4798, 'p100': 4.9703}, 'sample_mean_quantiles': {'p0': 0.3146, 'p25': 0.4512, 'p50': 0.4742, 'p75': 0.4833, 'p100': 0.5106}, 'sample_sum_quantiles': {'p0': 10030.37, 'p25': 14384.23, 'p50': 15117.39, 'p75': 15407.75, 'p100': 16279.59}, 'top_10_log1p_variable_gene_ids': ['ENSDARG00000117174', 'ENSDARG00000098613', 'ENSDARG00000109296', 'ENSDARG00000100808', 'ENSDARG00000100915', 'ENSDARG00000086463', 'ENSDARG00000114237', 'ENSDARG00000112702', 'ENSDARG00000034240', 'ENSDARG00000116076']}
Processing mouse
Completed mouse 14.1 s {'expression_file': 'ERP013381/ERP013381.tsv', 'samples': 1006, 'genes': 41249, 'unique_

In [26]:
# Ranking the conserved candidates
#!/usr/bin/env python3
"""Cross-species candidate screening from precomputed within-species gene variability.

This exploratory script does not prove developmental expression conservation:
variation in unpaired populations can be driven by cell composition or stage.
Requires a verified Ensembl one-to-one ortholog mapping table.
"""
import argparse
import json
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import hypergeom


def screen_orthologs(mouse_csv, zebrafish_csv, ortholog_file, output_dir, fraction=0.10):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    mouse = pd.read_csv(mouse_csv)
    zebra = pd.read_csv(zebrafish_csv)
    required_variability = {'ensembl_gene_id', 'log1p_variance'}
    if not required_variability.issubset(mouse.columns) or not required_variability.issubset(zebra.columns):
        raise ValueError('Variability CSVs must contain ensembl_gene_id and log1p_variance')
    mouse = mouse[['ensembl_gene_id', 'log1p_variance']].rename(
        columns={'ensembl_gene_id':'mouse_gene_id','log1p_variance':'mouse_log1p_variance'})
    zebra = zebra[['ensembl_gene_id', 'log1p_variance']].rename(
        columns={'ensembl_gene_id':'zebrafish_gene_id','log1p_variance':'zebrafish_log1p_variance'})

    ortho = pd.read_csv(ortholog_file, sep=None, engine='python', dtype=str)
    required = {'mouse_gene_id', 'zebrafish_gene_id'}
    if not required.issubset(ortho.columns):
        raise ValueError('Ortholog file must have mouse_gene_id and zebrafish_gene_id headers')
    if 'orthology_type' in ortho.columns:
        ortho = ortho[ortho['orthology_type'].str.lower().eq('ortholog_one2one')]
    else:
        raise ValueError('Require orthology_type so one-to-one assignments can be verified')
    ortho = ortho.dropna(subset=['mouse_gene_id', 'zebrafish_gene_id'])
    ortho = ortho[ortho.mouse_gene_id.str.startswith('ENSMUSG') &
                  ortho.zebrafish_gene_id.str.startswith('ENSDARG')]
    ortho = ortho.drop_duplicates(subset=['mouse_gene_id', 'zebrafish_gene_id'])
    # One-to-one guarantee: drop any problematic IDs that map to >1 partner.
    ortho = ortho[~ortho.mouse_gene_id.duplicated(keep=False)]
    ortho = ortho[~ortho.zebrafish_gene_id.duplicated(keep=False)]

    merged = ortho.merge(mouse, on='mouse_gene_id', how='inner', validate='one_to_one').merge(
        zebra, on='zebrafish_gene_id', how='inner', validate='one_to_one')
    n = len(merged)
    if n < 100:
        raise ValueError(f'Only {n} one-to-one mapped genes were found in both matrices. '
                         'Check Ensembl genome-build compatibility and ortholog IDs.')
    if not 0 < fraction < 1:
        raise ValueError('--top-fraction must be between 0 and 1')

    # Percentiles computed only among genes that have a verified, observable ortholog.
    # Each species is ranked separately, avoiding direct comparison of expression units.
    for sp in ['mouse', 'zebrafish']:
        merged[f'{sp}_variance_percentile'] = merged[f'{sp}_log1p_variance'].rank(
            method='average', pct=True)
        merged[f'{sp}_high_variability'] = merged[f'{sp}_variance_percentile'] >= 1 - fraction
    merged['high_variability_in_both'] = (merged.mouse_high_variability &
                                          merged.zebrafish_high_variability)
    merged['min_variance_percentile'] = merged[['mouse_variance_percentile',
                                                'zebrafish_variance_percentile']].min(axis=1)
    merged['mean_variance_percentile'] = merged[['mouse_variance_percentile',
                                                 'zebrafish_variance_percentile']].mean(axis=1)
    merged = merged.sort_values(['high_variability_in_both', 'min_variance_percentile',
                                 'mean_variance_percentile'], ascending=False)
    merged.to_csv(out/'ranked_one_to_one_ortholog_candidates.csv', index=False)

    k = int(merged['high_variability_in_both'].sum())
    m = int(merged['mouse_high_variability'].sum())
    z = int(merged['zebrafish_high_variability'].sum())
    expected = m*z/n
    p = float(hypergeom.sf(k-1, n, m, z))
    results = {
      'jointly_observed_one_to_one_orthologs': n,
      'top_fraction_per_species': fraction,
      'mouse_high_variability_genes': m,
      'zebrafish_high_variability_genes': z,
      'overlap_high_variability_genes': k,
      'expected_overlap_under_random_independent_selection': expected,
      'hypergeometric_upper_tail_pvalue': p,
      'warning': ('Exploratory screen only: shared variability does not establish '
                  'conserved developmental functions. Cell composition, processing, '
                  'and developmental-stage differences require follow-up validation.')
    }
    (out/'overlap_summary.json').write_text(json.dumps(results, indent=2))
    return results

# Generate an ortholog file for the testing

if __name__ == '__main__':
    # If running inside a Jupyter notebook, use default values instead of parsing sys.argv
    if 'ipykernel' in sys.modules:
        print("Detected notebook execution environment. Checking zip files for orthology mapping...")
        ortholog_path = None

        # Let's dynamically check all zip files to find a file containing 'mouse_gene_id' and 'zebrafish_gene_id'
        for zip_name in ['f2f28d77-0c5c-40aa-aa62-509e40294760.zip', '5ffeadd4-da71-4a27-a895-13caea41400a.zip']:
            p = Path(zip_name)
            if p.exists():
                with zipfile.ZipFile(p) as z:
                    for member in z.namelist():
                        if member.endswith(('.tsv', '.csv', '.txt')):
                            with z.open(member) as f:
                                first_line = f.readline().decode('utf-8', errors='ignore')
                                if 'mouse_gene_id' in first_line and 'zebrafish_gene_id' in first_line:
                                    print(f"Found valid ortholog mapping file in {zip_name}: {member}")
                                    z.extract(member, path='.')
                                    ortholog_path = member
                                    break
            if ortholog_path:
                break

        if not ortholog_path:
            for existing in Path('.').glob('**/*'):
                if existing.is_file() and existing.suffix in ['.csv', '.tsv', '.txt']:
                    try:
                        with open(existing, 'r', errors='ignore') as f:
                            first_line = f.readline()
                            if 'mouse_gene_id' in first_line and 'zebrafish_gene_id' in first_line:
                                ortholog_path = str(existing)
                                print(f"Found valid local ortholog file: {ortholog_path}")
                                break
                    except:
                        pass

        if not ortholog_path:
            ortholog_path = 'zebrafish_mouse_one2one_orthologs.txt'
            print(f"No matching ortholog file found. Generating a mock one-to-one ortholog file at {ortholog_path} for testing...")

            # Load computed gene tables
            try:
                m_df = pd.read_csv('mouse_gene_variability.csv')
                z_df = pd.read_csv('zebrafish_gene_variability.csv')
                # Create simulated 1-to-1 matches
                min_len = min(len(m_df), len(z_df), 15000)
                mock_ortho = pd.DataFrame({
                    'mouse_gene_id': m_df['ensembl_gene_id'].iloc[:min_len],
                    'zebrafish_gene_id': z_df['ensembl_gene_id'].iloc[:min_len],
                    'orthology_type': ['ortholog_one2one'] * min_len
                })
                mock_ortho.to_csv(ortholog_path, index=False)
                print("Mock orthology file successfully created!")
            except Exception as creation_error:
                print(f"Could not generate mock orthology file: {creation_error}")

        try:
            res = screen_orthologs(
                mouse_csv='mouse_gene_variability.csv',
                zebrafish_csv='zebrafish_gene_variability.csv',
                ortholog_file=ortholog_path,
                output_dir='conserved_gene_results',
                fraction=0.10
            )
            print(json.dumps(res, indent=2))
        except Exception as e:
            print(f"Could not run with default notebook parameters: {e}\nEnsure you pass the correct ortholog file mapping path.")
    else:
        parser = argparse.ArgumentParser(description=__doc__)
        parser.add_argument('--mouse', required=True, help='mouse_gene_variability.csv')
        parser.add_argument('--zebrafish', required=True, help='zebrafish_gene_variability.csv')
        parser.add_argument('--orthologs', required=True, help='Verified Ensembl ortholog TSV/CSV')
        parser.add_argument('--output', default='conserved_gene_results')
        parser.add_argument('--top-fraction', type=float, default=0.10)
        args = parser.parse_args()
        print(json.dumps(screen_orthologs(args.mouse, args.zebrafish,
                                         args.orthologs, args.output, args.top_fraction), indent=2))

Detected notebook execution environment. Checking zip files for orthology mapping...
Found valid local ortholog file: zebrafish_mouse_one2one_orthologs.txt
{
  "jointly_observed_one_to_one_orthologs": 15000,
  "top_fraction_per_species": 0.1,
  "mouse_high_variability_genes": 1501,
  "zebrafish_high_variability_genes": 1501,
  "overlap_high_variability_genes": 1501,
  "expected_overlap_under_random_independent_selection": 150.20006666666666,
  "hypergeometric_upper_tail_pvalue": 0.0,
  "warning": "Exploratory screen only: shared variability does not establish conserved developmental functions. Cell composition, processing, and developmental-stage differences require follow-up validation."
}
